# High-freq calls — clean bout analysis

Rewrite of `high_freq_bouts_v3.ipynb` using the shared `detect_bouts` helper. Thresholds come from `vocalization_analysis.bouts.BOUT_THRESHOLDS["high-freq"]`.

Note: `high_freq_bouts_v3` also did *merged* HF+warble bouts. This notebook starts with HF-only; we can add the merged version later if/when we need it.

We'll add analysis cells one at a time.

In [ ]:
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from vocalization_analysis.bouts import BOUT_THRESHOLDS, detect_bouts

HOST = platform.system()
if HOST == "Darwin":
    DROPBOX              = Path("/Users/gilyginosar/Dropbox (Personal)/Vocalizations_project")
    PARQUET_DIR          = DROPBOX / "Data" / "parquet_cache"
    FIGURES_DIR          = DROPBOX / "Figures" / "high_freq_bouts_clean"
    BASE_PROCESSED_AUDIO = None
    SAVE_FIGS            = True
elif HOST == "Linux":
    PARQUET_DIR          = Path("/mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data/Audio/all_calls/parquet_cache")
    BASE_PROCESSED_AUDIO = Path("/mnt/home/neurostatslab/ceph/saneslab_data/gily_data/Processed_data/Audio")
    FIGURES_DIR          = None
    SAVE_FIGS            = False
else:
    raise RuntimeError(f"Unsupported platform: {HOST}")

if SAVE_FIGS:
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(fig, name, fmt="pdf"):
    if not SAVE_FIGS:
        return
    fig.savefig(FIGURES_DIR / f"{name}.{fmt}", bbox_inches="tight")

CALL_TYPE     = "high-freq"
DATES_TO_PLOT = ["2025_03", "2025_07", "2025_10", "2026_02"]

print(f"HOST              = {HOST}")
print(f"PARQUET_DIR       = {PARQUET_DIR}")
print(f"SAVE_FIGS         = {SAVE_FIGS}")
print(f"call_type         = {CALL_TYPE}")
print(f"bout thresholds   = {BOUT_THRESHOLDS[CALL_TYPE]}")

In [ ]:
# Load + filter to one call type, then attach bout columns.
parts = []
for date_tag in DATES_TO_PLOT:
    df_d = pd.read_parquet(PARQUET_DIR / f"all_calls_{date_tag}.parquet")
    df_d = df_d[df_d["event_type"] == CALL_TYPE]
    parts.append(df_d)
calls = pd.concat(parts, ignore_index=True)

calls = detect_bouts(calls, CALL_TYPE)

MIN_BOUT_SIZE = BOUT_THRESHOLDS[CALL_TYPE]["min_bout_size"]

print(f"{len(calls):,} {CALL_TYPE} calls total")
print()
print("Calls per (date, bout_kind):")
print(calls.groupby(["date_folder", "bout_kind"]).size().unstack(fill_value=0))
print()
print("Bout-size summary per date (unique bouts):")
bouts = calls.drop_duplicates("bout_id")
print(bouts.groupby("date_folder")["bout_size"].agg(["count", "mean", "median", "max"]))